# Pitch Vision — Stage 2 (Tracking & Team Intelligence)

Turns per-frame detections into a coherent match: persistent player/ball IDs (ByteTrack), proper annotation drawing, automatic team-color clustering (with the referee filtered out by color, not forced into a team), ball-gap interpolation, and a possession stat.

This writes real project files (`utils/`, `trackers/`, `team_assigner/`, `player_ball_assigner/`) into Colab's filesystem so you can download them afterward and drop them straight into your local project — this is the actual pipeline code, not throwaway notebook cells.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough — this is inference, not training, much lighter than Stage 1).

In [ ]:
!pip install -q ultralytics supervision
import os
for d in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner']:
    os.makedirs(d, exist_ok=True)
print("Project folders created.")

In [ ]:
from google.colab import files
print("Upload best_topview.pt (your Stage 1 top-view fine-tuned model):")
uploaded = files.upload()
model_path = list(uploaded.keys())[0]
print("Using model:", model_path)

In [ ]:
print("Upload clip_01_0-12s.mp4 (from your project's input_videos/ folder):")
uploaded_clip = files.upload()
clip_path = list(uploaded_clip.keys())[0]
print("Using clip:", clip_path)

## Writing the pipeline modules

In [ ]:
%%writefile utils/__init__.py


In [ ]:
%%writefile trackers/__init__.py


In [ ]:
%%writefile team_assigner/__init__.py


In [ ]:
%%writefile player_ball_assigner/__init__.py


In [ ]:
%%writefile utils/bbox_utils.py
def get_center_of_bbox(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def get_bbox_width(bbox):
    return bbox[2] - bbox[0]


def get_foot_position(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int(y2)


In [ ]:
%%writefile utils/video_utils.py
import cv2


def read_video(video_path, target_width=1920):
    # Resize down while reading, not after — our clip is native 4K (3840x2160), and
    # holding all ~360 frames in memory at full 4K is ~9GB on its own, enough to crash
    # Colab's free-tier RAM by itself. target_width=1920 matches the resolution our
    # training data was stored at, so detection stays consistent with training too.
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        if target_width is not None and w > target_width:
            scale = target_width / w
            frame = cv2.resize(frame, (target_width, int(h * scale)), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()
    return frames


def get_native_frame(video_path, frame_idx):
    """
    Reads exactly ONE frame directly from the source video file at its original,
    un-downsampled resolution (e.g. native 4K even though read_video() gives back
    1920-wide frames for detection/tracking/drawing).

    Used only for team-color sampling: a player crop that's already tiny gets made
    even blurrier by the resize read_video() does to keep RAM usage sane, which
    contaminates shirt-color sampling with blended-in grass pixels. Re-reading just
    the handful of frames we actually need color from, at full detail, avoids that
    without holding the whole clip in memory at 4K.
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")
    return frame


def save_video(output_video_frames, output_video_path, fps=25):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    h, w = output_video_frames[0].shape[:2]
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
    for frame in output_video_frames:
        out.write(frame)
    out.release()


In [ ]:
%%writefile trackers/tracker.py
import os
import pickle
import sys

import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox, get_bbox_width


class Tracker:
    def __init__(self, model_path):
        self.model = YOLO(model_path)
        self.tracker = sv.ByteTrack()

    def detect_frames(self, frames, conf=0.2, imgsz=1280, batch_size=20):
        # imgsz=1280 matches training — inference at the default 640 would shrink the
        # ball back down below what the model was actually trained to recognize.
        detections = []
        for i in range(0, len(frames), batch_size):
            batch = self.model.predict(frames[i:i + batch_size], conf=conf, imgsz=imgsz, verbose=False)
            detections += batch
        return detections

    def get_object_tracks(self, frames, read_from_stub=False, stub_path=None):
        if read_from_stub and stub_path is not None and os.path.exists(stub_path):
            with open(stub_path, 'rb') as f:
                return pickle.load(f)

        detections = self.detect_frames(frames)

        tracks = {"players": [], "ball": []}

        for frame_num, detection in enumerate(detections):
            cls_names = detection.names  # {0: 'player', 1: 'ball'}
            cls_names_inv = {v: k for k, v in cls_names.items()}

            detection_supervision = sv.Detections.from_ultralytics(detection)
            detection_with_tracks = self.tracker.update_with_detections(detection_supervision)

            tracks["players"].append({})
            tracks["ball"].append({})

            # players get persistent track_ids from ByteTrack
            for frame_detection in detection_with_tracks:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                track_id = frame_detection[4]
                if cls_id == cls_names_inv.get('player'):
                    tracks["players"][frame_num][track_id] = {"bbox": bbox}

            # the ball gets a hardcoded track_id of 1 — there's only ever one, no need to
            # track its identity across frames, just its position
            for frame_detection in detection_supervision:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                if cls_id == cls_names_inv.get('ball'):
                    tracks["ball"][frame_num][1] = {"bbox": bbox}

        if stub_path is not None:
            with open(stub_path, 'wb') as f:
                pickle.dump(tracks, f)

        return tracks

    def draw_ellipse(self, frame, bbox, color, track_id=None):
        y2 = int(bbox[3])
        x_center, _ = get_center_of_bbox(bbox)
        width = get_bbox_width(bbox)

        cv2.ellipse(
            frame,
            center=(x_center, y2),
            axes=(int(width), int(0.35 * width)),
            angle=0.0,
            startAngle=-45,
            endAngle=235,
            color=color,
            thickness=2,
            lineType=cv2.LINE_4,
        )

        rect_w, rect_h = 40, 20
        x1_rect = x_center - rect_w // 2
        x2_rect = x_center + rect_w // 2
        y1_rect = (y2 - rect_h // 2) + 15
        y2_rect = (y2 + rect_h // 2) + 15

        if track_id is not None:
            cv2.rectangle(frame, (int(x1_rect), int(y1_rect)), (int(x2_rect), int(y2_rect)), color, cv2.FILLED)
            x1_text = x1_rect + 12
            if track_id > 99:
                x1_text -= 10
            cv2.putText(frame, f"{track_id}", (int(x1_text), int(y1_rect + 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        return frame

    def draw_triangle(self, frame, bbox, color):
        y = int(bbox[1])
        x, _ = get_center_of_bbox(bbox)
        points = np.array([[x, y], [x - 10, y - 20], [x + 10, y - 20]])
        cv2.drawContours(frame, [points], 0, color, cv2.FILLED)
        cv2.drawContours(frame, [points], 0, (0, 0, 0), 2)
        return frame

    def draw_annotations(self, video_frames, tracks, team_ball_control):
        output_frames = []
        for frame_num, frame in enumerate(video_frames):
            frame = frame.copy()
            player_dict = tracks["players"][frame_num]
            ball_dict = tracks["ball"][frame_num]

            for track_id, player in player_dict.items():
                color = player.get("team_color", (0, 0, 255))
                frame = self.draw_ellipse(frame, player["bbox"], color, track_id)
                if player.get("has_ball", False):
                    frame = self.draw_triangle(frame, player["bbox"], (0, 0, 255))

            for _, ball in ball_dict.items():
                frame = self.draw_triangle(frame, ball["bbox"], (0, 255, 0))

            if len(team_ball_control) > 0 and frame_num < len(team_ball_control):
                so_far = team_ball_control[:frame_num + 1]
                t1 = int((so_far == 1).sum())
                t2 = int((so_far == 2).sum())
                total = t1 + t2
                if total > 0:
                    cv2.putText(frame, f"Team 1 Ball Control: {t1 / total * 100:.1f}%", (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
                    cv2.putText(frame, f"Team 2 Ball Control: {t2 / total * 100:.1f}%", (50, 90),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

            output_frames.append(frame)
        return output_frames


In [ ]:
%%writefile team_assigner/team_assigner.py
import cv2
import numpy as np
from sklearn.cluster import KMeans


class TeamAssigner:
    """
    Assigns each tracked person to a team by clustering shirt color, and — since our
    detector only has a 'player' class (no separate 'referee' class in training data) —
    flags anyone whose shirt color doesn't cleanly match either team cluster as a
    non-player (referee) instead of forcing them into the nearest team.

    History of what didn't work, kept here because the next person touching this file
    (possibly future-me) will otherwise re-try the same dead ends:
      1. Top-half-of-box sampling (broadcast/side-view assumption: shirt on top, shorts
         below) -- wrong for an overhead view, where the top of a tiny box is head/hair.
      2. Full-box + inner 2-cluster KMeans, treating corner pixels as "background" --
         fails on tight boxes where the corners are still the player's own body.
      3. Plain median of the whole box, even the whole native-resolution box -- still
         failed in practice. Measured real output looked like BGR (91, 133, 109) and
         (101, 145, 124) for the two teams: nearly identical AND both green-dominant
         (G channel highest in both) -- a dead giveaway that grass pixels, not shirt
         pixels, were winning the median vote. A generously-sized/loosely-fit detection
         box around a small player can be majority background even when "tight" by eye.
    """

    # Grass in this footage (real turf, mowing stripes) sits in a fairly consistent
    # green hue band regardless of light/dark stripe -- stripes differ in brightness
    # (V), not hue. OpenCV hue is 0-179. Saturation gate avoids excluding dark/desaturated
    # shirts that merely happen to fall in the same hue range.
    GRASS_HUE_LOW = 25
    GRASS_HUE_HIGH = 95
    GRASS_SAT_MIN = 40

    def __init__(self):
        self.team_colors = {}
        self.player_team_dict = {}
        self.kmeans = None
        self.outlier_threshold = None

    def get_player_color(self, frame, bbox):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        image = frame[y1:y2, x1:x2]
        if image.size == 0:
            return np.array([0, 0, 0])

        # Trim a small margin off each edge -- the outermost pixels of even a tight box
        # are the most likely to be anti-aliased/motion-blurred blends with whatever's
        # just outside the player.
        h, w = image.shape[:2]
        my, mx = int(h * 0.1), int(w * 0.1)
        if h - 2 * my > 0 and w - 2 * mx > 0:
            image = image[my:h - my, mx:w - mx]

        # Explicitly drop grass-hued pixels before summarizing color. This is a more
        # direct fix than just hoping a tighter crop avoids background: it targets the
        # actual contamination (green pitch) by color, so it still works even when the
        # detection box itself is loose or the player is tiny and blurry.
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        hue, sat = hsv[:, :, 0], hsv[:, :, 1]
        is_grass = (hue >= self.GRASS_HUE_LOW) & (hue <= self.GRASS_HUE_HIGH) & (sat >= self.GRASS_SAT_MIN)

        pixels = image.reshape(-1, 3)
        keep = ~is_grass.reshape(-1)
        kept_pixels = pixels[keep]

        # If almost everything got excluded (box is nearly all pitch -- heavy occlusion,
        # a bad box, or a genuinely green/olive kit), fall back to the full crop rather
        # than return a median of a handful of pixels.
        if len(kept_pixels) < 0.15 * len(pixels):
            kept_pixels = pixels

        return np.median(kept_pixels, axis=0)

    def assign_team_colors_from_samples(self, player_colors_dict):
        """
        Lower-level entry point: takes a pre-computed {track_id: color} mapping (e.g.
        each player's color already aggregated/medianed across several frames by the
        caller) and does the actual team clustering + referee-outlier detection.
        Separated from assign_team_colors() so the pipeline can aggregate samples across
        multiple frames for stability instead of trusting a single frame.
        """
        track_ids = list(player_colors_dict.keys())
        player_colors = np.array([player_colors_dict[tid] for tid in track_ids])

        kmeans = KMeans(n_clusters=2, init="k-means++", n_init=10)
        kmeans.fit(player_colors)
        self.kmeans = kmeans

        self.team_colors[1] = kmeans.cluster_centers_[0]
        self.team_colors[2] = kmeans.cluster_centers_[1]

        # distance from each person's color to their nearest team-cluster center — a
        # referee's kit color won't match either team well, so this distance spikes for
        # them specifically. Threshold is data-driven (mean + 2*std), not a hardcoded guess.
        distances = []
        for color in player_colors:
            label = kmeans.predict(color.reshape(1, -1))[0]
            distances.append(np.linalg.norm(color - kmeans.cluster_centers_[label]))
        distances = np.array(distances)
        self.outlier_threshold = distances.mean() + 2 * distances.std() if len(distances) > 1 else np.inf

        for track_id, color in zip(track_ids, player_colors):
            label = kmeans.predict(color.reshape(1, -1))[0]
            dist = np.linalg.norm(color - kmeans.cluster_centers_[label])
            self.player_team_dict[track_id] = "referee" if dist > self.outlier_threshold else int(label) + 1

    def assign_team_colors(self, frame, player_detections):
        """Single-frame convenience wrapper around assign_team_colors_from_samples()."""
        player_colors = {
            track_id: self.get_player_color(frame, detection["bbox"])
            for track_id, detection in player_detections.items()
        }
        self.assign_team_colors_from_samples(player_colors)

    def get_player_team(self, frame, player_bbox, player_id):
        if player_id in self.player_team_dict:
            return self.player_team_dict[player_id]

        color = self.get_player_color(frame, player_bbox)
        label = self.kmeans.predict(color.reshape(1, -1))[0]
        dist = np.linalg.norm(color - self.kmeans.cluster_centers_[label])

        team = "referee" if (self.outlier_threshold is not None and dist > self.outlier_threshold) else int(label) + 1
        self.player_team_dict[player_id] = team
        return team


In [ ]:
%%writefile player_ball_assigner/player_ball_assigner.py
import sys

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox


class PlayerBallAssigner:
    def __init__(self, max_player_ball_distance=70):
        self.max_player_ball_distance = max_player_ball_distance

    def assign_ball_to_player(self, players, ball_bbox):
        ball_x, ball_y = get_center_of_bbox(ball_bbox)
        minimum_distance = float("inf")
        assigned_player = -1

        for player_id, player in players.items():
            bbox = player["bbox"]
            distance_left = ((bbox[0] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance_right = ((bbox[2] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance = min(distance_left, distance_right)

            if distance < self.max_player_ball_distance and distance < minimum_distance:
                minimum_distance = distance
                assigned_player = player_id

        return assigned_player


## Run the pipeline

Detect + track → interpolate the ball through gaps → cluster shirt colors into two teams (referee gets filtered out automatically) → figure out who has the ball each frame → draw the annotated video.

In [ ]:
import sys
sys.path.append('/content')
import numpy as np
import pandas as pd

from utils.video_utils import read_video, save_video, get_native_frame
from trackers.tracker import Tracker
from team_assigner.team_assigner import TeamAssigner
from player_ball_assigner.player_ball_assigner import PlayerBallAssigner

video_frames = read_video(clip_path)
print(f"Loaded {len(video_frames)} frames")

tracker = Tracker(model_path)
tracks = tracker.get_object_tracks(video_frames)
print("Detection + tracking done")

In [ ]:
# Ball position interpolation — fills gaps (occlusion, missed detections) frame by frame.
ball_positions = [x.get(1, {}).get("bbox", [np.nan] * 4) for x in tracks["ball"]]
df_ball = pd.DataFrame(ball_positions, columns=["x1", "y1", "x2", "y2"])
df_ball = df_ball.interpolate().bfill()
tracks["ball"] = [{1: {"bbox": row}} for row in df_ball.to_numpy().tolist()]
print("Ball positions interpolated")

In [ ]:
# Team assignment. Three stacked fixes, all aimed at the same root problem: on a
# full-pitch wide shot each player is only ~10-20px in the downsampled frames used for
# detection, and that's not enough real shirt-color signal to work with reliably.
#   1. Sample color from the ORIGINAL native-resolution video, not the downsampled
#      frames — far more real pixels per player.
#   2. Inside each crop, explicitly drop grass-hued pixels before summarizing color
#      (see team_assigner.py) — targets pitch contamination directly rather than
#      hoping a tighter/bigger crop avoids it.
#   3. Aggregate each player's color across several frames (not just frame 0), median
#      of medians — so one bad frame (motion blur, shadow, partial occlusion) can't
#      define that player's whole classification.
native_frame0 = get_native_frame(clip_path, 0)
scale = native_frame0.shape[1] / video_frames[0].shape[1]
print(f"Native resolution is {scale:.2f}x the downsampled frames used for detection")

def _scale_bbox(bbox, s):
    return [c * s for c in bbox]

team_assigner = TeamAssigner()

n_frames = len(tracks["players"])
sample_frame_nums = sorted(set(min(n_frames - 1, int(n_frames * f)) for f in [0.0, 0.2, 0.4, 0.6, 0.8]))
print(f"Sampling team color from frames: {sample_frame_nums}")

per_player_samples = {}
for fn in sample_frame_nums:
    native = get_native_frame(clip_path, fn)
    for player_id, track in tracks["players"][fn].items():
        color = team_assigner.get_player_color(native, _scale_bbox(track["bbox"], scale))
        per_player_samples.setdefault(player_id, []).append(color)

aggregate_colors = {tid: np.median(np.array(colors), axis=0) for tid, colors in per_player_samples.items()}
print(f"Aggregated color for {len(aggregate_colors)} distinct players across {len(sample_frame_nums)} sample frames")

team_assigner.assign_team_colors_from_samples(aggregate_colors)

# Anyone whose shirt color doesn't match either team cluster — i.e. the referee — gets
# tagged "referee" instead of being forced into a team. Most players are already cached
# from the sampling pass above; only a brand-new track_id (never seen in any sample
# frame) needs its own native-resolution lookup here, done lazily.
for frame_num, player_track in enumerate(tracks["players"]):
    native_frame = None
    for player_id, track in player_track.items():
        if player_id in team_assigner.player_team_dict:
            team = team_assigner.player_team_dict[player_id]
        else:
            if native_frame is None:
                native_frame = get_native_frame(clip_path, frame_num)
            team = team_assigner.get_player_team(native_frame, _scale_bbox(track["bbox"], scale), player_id)

        tracks["players"][frame_num][player_id]["team"] = team
        if team == "referee":
            tracks["players"][frame_num][player_id]["team_color"] = (255, 255, 255)
        else:
            tracks["players"][frame_num][player_id]["team_color"] = tuple(
                int(c) for c in team_assigner.team_colors[team]
            )

print("Team colors (BGR):", team_assigner.team_colors)
print("Outlier threshold:", team_assigner.outlier_threshold)
referee_count = sum(1 for p in tracks["players"][0].values() if p.get("team") == "referee")
print(f"Frame 0: {len(tracks['players'][0])} people detected, {referee_count} flagged as referee")

In [ ]:
# Possession: nearest player to the ball each frame (within a distance threshold), plus a
# running team ball-control percentage.
ball_assigner = PlayerBallAssigner()
team_ball_control = []
for frame_num, player_track in enumerate(tracks["players"]):
    ball_bbox = tracks["ball"][frame_num][1]["bbox"]
    assigned_player = ball_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1:
        tracks["players"][frame_num][assigned_player]["has_ball"] = True
        assigned_team = tracks["players"][frame_num][assigned_player].get("team")
        if assigned_team in (1, 2):
            team_ball_control.append(assigned_team)
        else:
            team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)
    else:
        team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)

team_ball_control = np.array(team_ball_control)
print("Possession computed")
print(f"Team 1 final possession: {(team_ball_control == 1).sum() / len(team_ball_control) * 100:.1f}%")
print(f"Team 2 final possession: {(team_ball_control == 2).sum() / len(team_ball_control) * 100:.1f}%")

In [ ]:
output_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)
save_video(output_frames, 'stage2_output.mp4', fps=25)
print("Saved stage2_output.mp4")
files.download('stage2_output.mp4')

## Bring the code home

Download the actual pipeline files to drop into your local project folder — these are your real Stage 2 modules, not throwaway notebook code.

In [ ]:
import zipfile, os

with zipfile.ZipFile('stage2_modules.zip', 'w') as z:
    for folder in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner']:
        for root, _, filenames in os.walk(folder):
            for fn in filenames:
                z.write(os.path.join(root, fn))
files.download('stage2_modules.zip')
print("Downloaded stage2_modules.zip — unzip into your local project folder root.")

### Done for now

You should have downloaded: `stage2_output.mp4` (the annotated video — team-colored ellipses, track IDs, ball marker, possession % overlay) and `stage2_modules.zip` (the actual pipeline code: `utils/`, `trackers/`, `team_assigner/`, `player_ball_assigner/`).

Watch the output video and check: are the two teams colored consistently and correctly? Does the referee stay white/unteamed instead of flickering between team colors? Does the possession % look plausible given what you saw in the clip?